# 02 — Feature Engineering

**Goal:** Build the master feature table by merging all Home Credit tables
and deriving behavioral, demographic, and credit bureau features.

## Contents
1. Load all raw tables
2. Application-level feature engineering
3. Installment payment behavioral features
4. Credit card balance features
5. Bureau history features
6. Previous application features
7. Merge into master table
8. Feature importance preview (mutual information)

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

from features import (
    build_application_features, build_bureau_features,
    build_previous_app_features, build_installment_features,
    build_cc_features, build_pos_features,
)
from utils.config import load_config

sns.set_theme(style='whitegrid')
cfg = load_config()

## 1. Load Raw Tables

In [ ]:
app   = pd.read_csv(cfg.data.application_train)
bur   = pd.read_csv(cfg.data.bureau)
bb    = pd.read_csv(cfg.data.bureau_balance)
prev  = pd.read_csv(cfg.data.previous_application)
inst  = pd.read_csv(cfg.data.installments_payments)
cc    = pd.read_csv(cfg.data.credit_card_balance)
pos   = pd.read_csv(cfg.data.pos_cash_balance)

print('Tables loaded:')
for name, df in [('app',app),('bureau',bur),('bureau_bal',bb),
                  ('prev',prev),('inst',inst),('cc',cc),('pos',pos)]:
    print(f'  {name}: {df.shape}')

## 2. Application Features

In [ ]:
app_feats = build_application_features(app)
new_cols = set(app_feats.columns) - set(app.columns)
print(f'New features added: {sorted(new_cols)}')
app_feats[list(new_cols)[:6]].head(3)

## 3. Behavioral Features — Installment Payments

These capture *how consistently* borrowers make their payments, including:
- Days late (mean/max/std)
- DPD30 / DPD60 / DPD90 flags
- On-time payment velocity

In [ ]:
inst_feats = build_installment_features(inst)
print('Installment features:', inst_feats.shape)

# Visualise DPD30 rate distribution by default status
temp = inst_feats.merge(app[['SK_ID_CURR', 'TARGET']], on='SK_ID_CURR')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t, color, label in [(0,'#2ecc71','Non-Default'), (1,'#e74c3c','Default')]:
    axes[0].hist(temp.loc[temp['TARGET']==t, 'inst_days_late_mean'].clip(-30,60),
                  bins=50, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Mean Days Late Distribution', fontweight='bold')
axes[0].legend()

axes[1].bar(['Non-Default','Default'],
            [temp.loc[temp['TARGET']==0,'inst_dpd30_rate'].mean(),
             temp.loc[temp['TARGET']==1,'inst_dpd30_rate'].mean()],
            color=['#2ecc71','#e74c3c'])
axes[1].set_title('Average DPD30 Rate', fontweight='bold')
axes[1].set_ylabel('Rate')
plt.tight_layout()
plt.savefig('../outputs/figures/02_behavioral_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Bureau & Previous Application Features

In [ ]:
bureau_feats = build_bureau_features(bur, bb)
prev_feats   = build_previous_app_features(prev)
cc_feats     = build_cc_features(cc)
pos_feats    = build_pos_features(pos)

print('Bureau features:',   bureau_feats.shape)
print('Previous app feats:',prev_feats.shape)
print('CC balance feats:',  cc_feats.shape)
print('POS cash feats:',    pos_feats.shape)

## 5. Merge into Master Table

In [ ]:
master = (
    app_feats
    .merge(bureau_feats,  on='SK_ID_CURR', how='left')
    .merge(prev_feats,    on='SK_ID_CURR', how='left')
    .merge(inst_feats,    on='SK_ID_CURR', how='left')
    .merge(cc_feats,      on='SK_ID_CURR', how='left')
    .merge(pos_feats,     on='SK_ID_CURR', how='left')
)

print(f'Master table: {master.shape}')
print(f'Default rate preserved: {master["TARGET"].mean():.2%}')
master.head(3)

## 6. Feature Importance Preview (Mutual Information)

In [ ]:
# Numeric features only, drop high-missing
numeric = master.select_dtypes(include='number').copy()
numeric = numeric.drop(columns=['SK_ID_CURR'], errors='ignore')
numeric = numeric.loc[:, numeric.isnull().mean() < 0.5]
numeric = numeric.fillna(numeric.median())

y = numeric.pop('TARGET')
mi = mutual_info_classif(numeric, y, random_state=42)
mi_series = pd.Series(mi, index=numeric.columns).sort_values(ascending=False)

top20 = mi_series.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1],
        color=sns.color_palette('Blues_d', len(top20)))
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 20 Features by Mutual Information', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/02_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

# Save master
import os
os.makedirs('../data/processed/feature_cache', exist_ok=True)
master.to_csv('../data/processed/feature_cache/master_features.csv', index=False)
print('Master features saved.')